In [1]:
import os
import sys

import numpy as np
from tqdm import tqdm
from glob import glob
import matplotlib.pyplot as plt

### Play around

In [89]:
from cs336_basics.tokenizer import Tokenizer

data_dir = "/scratch/shared/beegfs/piyush/datasets/text_data"
dataset = "TinyStoriesV2-GPT4-train"
dataset = "owt_train"
filepath = f"{data_dir}/{dataset}-bpe_optimized.npz"
textpath = f"{data_dir}/{dataset}.txt"

tokenizer = Tokenizer.from_file(filepath=filepath, special_tokens=['<|endoftext|>'])
tokenizer

Showing random merged tokens from the vocabulary:
[ harm][ful] -> [ harmful] 12759
[ D][i] -> [ Di] 6654
[ M][ich] -> [ Mich] 2532
[ prov][ided] -> [ provided] 3334
[ V][II] -> [ VII] 24383
[ Cle][arly] -> [ Clearly] 21145
[ rout][e] -> [ route] 5878
[h][uman] -> [human] 10410
[ F][rench] -> [ French] 3664
[ v][oc] -> [ voc] 10984


Tokenizer(vocab=32257, merges=32000, special_tokens=['<|endoftext|>'])

In [90]:
tokenizer.vocab[len(tokenizer.vocab) - 1], tokenizer.PAT

(b'<|endoftext|>',
 regex.Regex("'(?:[sdmt]|ll|ve|re)| ?\\p{L}+| ?\\p{N}+| ?[^\\s\\p{L}\\p{N}]+|\\s+(?!\\S)|\\s+", flags=regex.V0))

In [97]:
text = "hello<|endoftext|>world<|endoftext|>bye. this is a bad suffice."
# text = """A dog! is a great animal."""
# text = "For example, suppose our input string is 'the cat ate', our vocabulary is"
print(text)
tokens = tokenizer.encode(text)
print(tokens)
reco = tokenizer.decode(tokens)
print(reco)
assert text == reco

hello<|endoftext|>world<|endoftext|>bye. this is a bad suffice.
[31359, 32256, 7660, 32256, 16575, 46, 431, 320, 257, 1978, 2428, 501, 46]
hello<|endoftext|>world<|endoftext|>bye. this is a bad suffice.


In [96]:
comp_ratio = len(list(text.encode("utf-8"))) / len(tokens)
comp_ratio

4.5625

### Experiments

In [108]:
data_dir = "/scratch/shared/beegfs/piyush/datasets/text_data"

dataset = "TinyStoriesV2-GPT4-train"
# dataset = "owt_train"
filepath = f"{data_dir}/{dataset}-bpe_optimized.npz"

dataset = "owt_train"
# dataset = "TinyStoriesV2-GPT4-train"
textpath = f"{data_dir}/{dataset}.txt"

# Initialize tokenizer
tokenizer = Tokenizer.from_file(filepath=filepath, special_tokens=['<|endoftext|>'])

In [109]:
import os
from typing import Iterable, Iterator, Sequence
import regex as re

def iter_text_chunks(
    path: str,
    special_tokens: Sequence[str],
    num_boundaries: int = 1024,
    encoding: str = "utf-8",
) -> Iterator[str]:
    """
    Yield chunks of text from a file without loading the whole file into memory.
    Chunk boundaries are moved forward until one of the special tokens is found.

    Parameters
    ----------
    path : str
        Path to the text file.
    special_tokens : Sequence[str]
        Special tokens to split on, e.g. ["<|endoftext|>"].
    num_boundaries : int
        Desired number of chunks.
    encoding : str
        Text encoding used to decode chunks.

    Yields
    ------
    str
        One decoded text chunk at a time.
    """
    token_bytes = [tok.encode(encoding) for tok in special_tokens]
    if not token_bytes:
        raise ValueError("special_tokens must not be empty")

    with open(path, "rb") as f:
        # File size
        f.seek(0, os.SEEK_END)
        file_size = f.tell()
        f.seek(0)

        if file_size == 0:
            return

        # Initial equally spaced guesses
        num_boundaries = max(1, num_boundaries)
        chunk_size = max(1, file_size // num_boundaries)
        boundaries = [i * chunk_size for i in range(num_boundaries)]
        boundaries.append(file_size)
        boundaries[-1] = file_size

        mini_chunk_size = 4096

        # Move each internal boundary forward until a special token is found
        for i in range(1, len(boundaries) - 1):
            pos = boundaries[i]
            f.seek(pos)

            while True:
                block = f.read(mini_chunk_size)
                if block == b"":
                    boundaries[i] = file_size
                    break

                found_positions = [
                    block.find(tok) for tok in token_bytes if block.find(tok) != -1
                ]

                if found_positions:
                    boundaries[i] = pos + min(found_positions)
                    break

                pos += mini_chunk_size

        # Remove duplicates and keep order
        boundaries = sorted(set(boundaries))

        # Yield chunks one at a time
        for start, end in zip(boundaries[:-1], boundaries[1:]):
            if start >= end:
                continue
            f.seek(start)
            raw = f.read(end - start)
            yield raw.decode(encoding, errors="ignore")


import psutil
process = psutil.Process(os.getpid())

iterable = iter_text_chunks(
    textpath,
    special_tokens=["<|endoftext|>"],
    num_boundaries=1024,
)
i = 0
docs = []
for chunk in tqdm(iterable, desc="Processing chunks", total=1024):
    if i == 0:
        before = process.memory_info().rss
        chunk_size = len(chunk.encode("utf-8"))
        after = process.memory_info().rss
        print("Memory for a single chunk:", sys.getsizeof(chunk) / 1e6, "MB")
        print(f"chunk_bytes={chunk_size/1e6}, rss_before={before/1e6}, rss_after={after/1e6}")
    i += 1
    docs.extend(chunk.split("<|endoftext|>"))
    if len(docs) > 100:
        break
    pass
    # print("----- Example chunk ------")
    # print(chunk[:2000])
    # print("--------------------------")
    # print("Length of chunk: ", len(chunk))
docs = np.random.choice(docs, 100)
len(docs)

Processing chunks:   0%|                                                                                                                                                                                                  | 0/1024 [00:01<?, ?it/s]

Memory for a single chunk: 46.161024 MB
chunk_bytes=11.644664, rss_before=281.68192, rss_after=281.68192


100

In [110]:
comp_ratios = []
for d in docs:
    t = tokenizer.encode(d)
    cr = len(d.encode("utf-8")) / len(t)
    comp_ratios.append(cr)
np.mean(comp_ratios)

np.float64(3.2873837135662267)

In [112]:
"<|endoftext|>" in chunk

True

In [113]:
matches = re.findall(re.compile(r"<\|\w+\|>"), chunk)
np.unique(matches), len(matches)

(array(['<|endoftext|>'], dtype='<U13'), 2346)

**Time estimation (bytes/sec)**

In [116]:
import random

def random_substrings(text, n=1000, min_len=500, max_len=5000):
    substrings = []
    L = len(text)

    for _ in range(n):
        start = random.randint(0, L - 1)
        length = random.randint(min_len, max_len)
        end = min(start + length, L)
        substrings.append(text[start:end])

    return substrings


substrings = random_substrings(chunk)
len(substrings)

1000

In [125]:
import time

bps = []
for s in tqdm(substrings):
    st = time.time()
    t = tokenizer.encode(s)
    et = time.time()
    nbytes = len(list(s.encode("utf=8")))
    nsec = (et - st)
    bps.append(nbytes / nsec)
print(len(bps), np.mean(bps), np.std(bps))

avg_gbps = np.mean(bps) / 2**30 # GB/s
print("Average GB/s: ", avg_gbps)

time_for_pile_in_s = (825. / avg_gbps)
time_for_pile_in_hours = time_for_pile_in_s / 3600.
print(f"Est. time to process The Pile: {time_for_pile_in_s}s ({time_for_pile_in_hours:.2f} hours = {time_for_pile_in_hours/24} days)")

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:08<00:00, 124.55it/s]

1000 353086.418801979 25284.547233992544
Average GB/s:  0.00032883735262041817
Est. time to process The Pile: 2508839.076296511s (696.90 hours = 29.0374893089874 days)


In [122]:
time_for_owt_hours = (13. / (avg_gbps * 3600))
time_for_owt_hours

np.float64(11.013545168748225)

In [124]:
time_for_ts_hours = (2.5 / (avg_gbps * 3600))
time_for_ts_hours

np.float64(2.117989455528505)